# Learning without Forgetting (LwF): ImageNet → CUB-200-2011

This notebook reproduces the ImageNet → CUB experiment using an AlexNet-based pipeline.

## Fixed protocol

- One run only: `seed = 42`
- ImageNet old-task evaluation: the first **80 validation batches**
- ImageNet batch size: `64`
- ImageNet loader: `shuffle=False`
- New-task training: **5 epochs** for each compared method
- Shared warm-up before Fine-Tuning and LwF: **1 epoch**
- CUB official test set: evaluated in full
- Internal validation split: used only for best-checkpoint selection
- Official CUB test images are never used for model selection

The notebook compares:

1. Fine-Tuning
2. Feature Extraction
3. Learning without Forgetting (LwF)


# Step 0 — Environment Setup


In [1]:
import os
import gc
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from PIL import Image, ImageFile
from IPython.display import display
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, models, transforms

ImageFile.LOAD_TRUNCATED_IMAGES = True

# -----------------------------
# Experiment configuration
# -----------------------------

# Exactly one complete run.
RUN_SEEDS = [42]

VAL_SPLIT_SEED = 123
VAL_FRACTION = 0.15

BATCH_SIZE_CUB = 32
BATCH_SIZE_IMAGENET = 64

# ImageNet is evaluated on the first 80 validation batches,
# consistently with the Scenes experiment.
IMAGENET_NUM_BATCHES = 80
IMAGENET_LOADER_SEED = 2026

NUM_WORKERS = 2

# Optimization settings
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005

BASE_LR = 0.001
HEAD_ONLY_LR = 0.005
FEATURE_EXTRACTION_LR = 0.005

TEMPERATURE = 2.0
LAMBDA_OLD = 1.0

# Exact requested training length
WARMUP_EPOCHS = 1
FINETUNING_EPOCHS = 5
FEATURE_EXTRACTION_EPOCHS = 5
LWF_EPOCHS = 5

SCHEDULER_PATIENCE = 1
SAVE_MODELS = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)
print("Run seeds:", RUN_SEEDS)
print("Warm-up epochs:", WARMUP_EPOCHS)
print("Fine-Tuning epochs:", FINETUNING_EPOCHS)
print("Feature Extraction epochs:", FEATURE_EXTRACTION_EPOCHS)
print("LwF epochs:", LWF_EPOCHS)
print("ImageNet evaluation batches:", IMAGENET_NUM_BATCHES)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Warning: a GPU is strongly recommended.")


PyTorch: 2.11.0+cpu
Device: cpu
Run seeds: [42]
Warm-up epochs: 1
Fine-Tuning epochs: 5
Feature Extraction epochs: 5
LwF epochs: 5
ImageNet evaluation batches: 80


# Step 1 — Mount Google Drive and Define Paths

The paths below preserve the folder structure used in the previous CUB notebook.  
Change only these path variables if your Google Drive folders are located elsewhere.


In [2]:
from google.colab import drive

drive.mount("/content/drive")

CUB_ROOT = "/content/drive/MyDrive/LWF_Project/extracted/LWF_Project/CUB_200_2011"
IMAGENET_ROOT = "/content/drive/MyDrive/lwf/imagenet_validation/imagenet_validation"
OUTPUT_ROOT = "/content/drive/MyDrive/LWF_Project/results_cub_consistent"

required_paths = [
    CUB_ROOT,
    os.path.join(CUB_ROOT, "images"),
    os.path.join(CUB_ROOT, "images.txt"),
    os.path.join(CUB_ROOT, "image_class_labels.txt"),
    os.path.join(CUB_ROOT, "train_test_split.txt"),
    IMAGENET_ROOT
]

for required_path in required_paths:
    if not os.path.exists(required_path):
        raise FileNotFoundError(
            f"Required path was not found: {required_path}\n"
            "Update the path variables in Step 1 before continuing."
        )

print("CUB root:", CUB_ROOT)
print("ImageNet validation root:", IMAGENET_ROOT)
print("Output root:", OUTPUT_ROOT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CUB root: /content/drive/MyDrive/LWF_Project/extracted/LWF_Project/CUB_200_2011
ImageNet validation root: /content/drive/MyDrive/lwf/imagenet_validation/imagenet_validation
Output root: /content/drive/MyDrive/LWF_Project/results_cub_consistent


# Step 2 — Read CUB Metadata and Preserve the Official Split

The official CUB train/test split is loaded from `train_test_split.txt`.

A stratified internal validation split is then created **only inside the official training set**.  
The official CUB test set remains untouched until final evaluation.


In [3]:
images_df = pd.read_csv(
    os.path.join(CUB_ROOT, "images.txt"),
    sep=" ",
    names=["image_id", "filepath"]
)

labels_df = pd.read_csv(
    os.path.join(CUB_ROOT, "image_class_labels.txt"),
    sep=" ",
    names=["image_id", "label"]
)

split_df = pd.read_csv(
    os.path.join(CUB_ROOT, "train_test_split.txt"),
    sep=" ",
    names=["image_id", "is_train"]
)

cub_metadata = (
    images_df
    .merge(labels_df, on="image_id")
    .merge(split_df, on="image_id")
)

# Convert CUB labels from 1..200 to 0..199.
cub_metadata["label_idx"] = cub_metadata["label"] - 1

official_train_df = (
    cub_metadata[cub_metadata["is_train"] == 1]
    .copy()
    .reset_index(drop=True)
)

official_test_df = (
    cub_metadata[cub_metadata["is_train"] == 0]
    .copy()
    .reset_index(drop=True)
)

NUM_CUB_CLASSES = int(cub_metadata["label_idx"].nunique())

print("Total CUB images:", len(cub_metadata))
print("Official CUB training images:", len(official_train_df))
print("Official CUB test images:", len(official_test_df))
print("CUB classes:", NUM_CUB_CLASSES)

if NUM_CUB_CLASSES != 200:
    raise RuntimeError(
        f"Expected 200 CUB classes, but found {NUM_CUB_CLASSES}."
    )


Total CUB images: 11788
Official CUB training images: 5994
Official CUB test images: 5794
CUB classes: 200


In [4]:
def stratified_dataframe_split(dataframe, label_column, val_fraction, seed):
    rng = np.random.default_rng(seed)

    train_indices = []
    val_indices = []

    for _, group in dataframe.groupby(label_column, sort=True):
        indices = group.index.to_numpy().copy()
        rng.shuffle(indices)

        num_val = max(1, int(round(len(indices) * val_fraction)))
        num_val = min(num_val, len(indices) - 1)

        val_indices.extend(indices[:num_val])
        train_indices.extend(indices[num_val:])

    train_subset = (
        dataframe.loc[train_indices]
        .sample(frac=1.0, random_state=seed)
        .reset_index(drop=True)
    )

    val_subset = (
        dataframe.loc[val_indices]
        .sample(frac=1.0, random_state=seed)
        .reset_index(drop=True)
    )

    return train_subset, val_subset


train_df, val_df = stratified_dataframe_split(
    dataframe=official_train_df,
    label_column="label_idx",
    val_fraction=VAL_FRACTION,
    seed=VAL_SPLIT_SEED
)

train_paths = set(train_df["filepath"])
val_paths = set(val_df["filepath"])
test_paths = set(official_test_df["filepath"])

assert train_paths.isdisjoint(val_paths)
assert train_paths.isdisjoint(test_paths)
assert val_paths.isdisjoint(test_paths)

print("Internal CUB training images:", len(train_df))
print("Internal CUB validation images:", len(val_df))
print("Official CUB test images:", len(official_test_df))
print("No official test image is used for checkpoint selection.")


Internal CUB training images: 5194
Internal CUB validation images: 800
Official CUB test images: 5794
No official test image is used for checkpoint selection.


# Step 3 — Define CUB Datasets and Data Augmentation

Training augmentation reduces overfitting:

- resize to `256 × 256`
- random crop to `224 × 224`
- random horizontal flip

Validation and testing use deterministic center crops.


In [5]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

cub_train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

cub_eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])


class CUBDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.dataframe = dataframe.reset_index(drop=True).copy()
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_path = os.path.join(
            self.root_dir,
            "images",
            row["filepath"]
        )

        if not os.path.isfile(image_path):
            raise FileNotFoundError(
                f"CUB image file was not found: {image_path}"
            )

        image = Image.open(image_path).convert("RGB")
        label = int(row["label_idx"])

        if self.transform is not None:
            image = self.transform(image)

        return image, label


train_dataset = CUBDataset(
    dataframe=train_df,
    root_dir=CUB_ROOT,
    transform=cub_train_transform
)

val_dataset = CUBDataset(
    dataframe=val_df,
    root_dir=CUB_ROOT,
    transform=cub_eval_transform
)

test_dataset = CUBDataset(
    dataframe=official_test_df,
    root_dir=CUB_ROOT,
    transform=cub_eval_transform
)

sample_image, sample_label = train_dataset[0]

print("Sample tensor shape:", sample_image.shape)
print("Sample label:", sample_label)


Sample tensor shape: torch.Size([3, 224, 224])
Sample label: 17


# Step 4 — Prepare the ImageNet Validation Dataset

ImageNet old-task accuracy is measured on the first **80 validation batches** with:

- `batch_size = 64`
- `shuffle = False`

This matches the Scenes experiment protocol exactly.


In [6]:
imagenet_eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

imagenet_full_dataset = datasets.ImageFolder(
    root=IMAGENET_ROOT,
    transform=imagenet_eval_transform
)

print("ImageNet validation images:", len(imagenet_full_dataset))
print("ImageNet classes:", len(imagenet_full_dataset.classes))
print("ImageNet evaluation batches:", IMAGENET_NUM_BATCHES)

if len(imagenet_full_dataset.classes) != 1000:
    raise RuntimeError(
        "Expected ImageNet validation folders for 1000 classes."
    )


ImageNet validation images: 50000
ImageNet classes: 1000
ImageNet evaluation batches: 80


# Step 5 — Reproducibility and DataLoader Helpers

A deterministic loader is created for every training stage.  
All compared methods use the same CUB split and the same evaluation protocol.


In [7]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


def make_loader(dataset, batch_size, shuffle, seed):
    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        worker_init_fn=seed_worker,
        generator=generator
    )


def make_cub_loaders(seed):
    train_loader = make_loader(
        dataset=train_dataset,
        batch_size=BATCH_SIZE_CUB,
        shuffle=True,
        seed=seed
    )

    val_loader = make_loader(
        dataset=val_dataset,
        batch_size=BATCH_SIZE_CUB,
        shuffle=False,
        seed=seed
    )

    test_loader = make_loader(
        dataset=test_dataset,
        batch_size=BATCH_SIZE_CUB,
        shuffle=False,
        seed=seed
    )

    return train_loader, val_loader, test_loader


imagenet_loader = make_loader(
    dataset=imagenet_full_dataset,
    batch_size=BATCH_SIZE_IMAGENET,
    shuffle=False,
    seed=IMAGENET_LOADER_SEED
)

print("Internal CUB train batches:", len(make_cub_loaders(42)[0]))
print("Internal CUB validation batches:", len(make_cub_loaders(42)[1]))
print("Official CUB test batches:", len(make_cub_loaders(42)[2]))
print("ImageNet full validation batches:", len(imagenet_loader))


Internal CUB train batches: 163
Internal CUB validation batches: 25
Official CUB test batches: 182
ImageNet full validation batches: 782


# Step 6 — Model Definitions

All compared methods use AlexNet.

## Dual-head design

- `old_head`: ImageNet classifier with 1,000 outputs
- `new_head`: CUB classifier with 200 outputs
- shared representation: all AlexNet layers before the original final classifier

## Feature Extraction design

The ImageNet representation is frozen.  
Dropout is disabled in the frozen extractor.  
A paper-inspired two-layer CUB head is trained:

`4096 → 4096 → 200`


In [8]:
def initialize_linear_layers(module):
    for layer in module.modules():
        if isinstance(layer, nn.Linear):
            nn.init.xavier_uniform_(layer.weight)

            if layer.bias is not None:
                nn.init.zeros_(layer.bias)


def set_requires_grad(module, requires_grad):
    for parameter in module.parameters():
        parameter.requires_grad = requires_grad


def create_pretrained_alexnet():
    return models.alexnet(
        weights=models.AlexNet_Weights.IMAGENET1K_V1
    )


class DualHeadAlexNet(nn.Module):
    def __init__(self, pretrained_model, num_new_classes):
        super().__init__()

        self.features = copy.deepcopy(pretrained_model.features)
        self.avgpool = copy.deepcopy(pretrained_model.avgpool)

        self.shared_classifier = copy.deepcopy(
            nn.Sequential(*list(pretrained_model.classifier.children())[:-1])
        )

        self.old_head = copy.deepcopy(pretrained_model.classifier[6])

        self.new_head = nn.Linear(4096, num_new_classes)
        initialize_linear_layers(self.new_head)

    def encode(self, images):
        x = self.features(images)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.shared_classifier(x)
        return x

    def forward(self, images, task):
        x = self.encode(images)

        if task == "imagenet":
            return self.old_head(x)

        if task == "cub":
            return self.new_head(x)

        raise ValueError("task must be either 'imagenet' or 'cub'")

    def forward_both(self, images):
        x = self.encode(images)
        return self.old_head(x), self.new_head(x)


class FeatureExtractionAlexNet(nn.Module):
    def __init__(self, pretrained_model, num_new_classes):
        super().__init__()

        self.features = copy.deepcopy(pretrained_model.features)
        self.avgpool = copy.deepcopy(pretrained_model.avgpool)

        self.shared_classifier = copy.deepcopy(
            nn.Sequential(*list(pretrained_model.classifier.children())[:-1])
        )

        # Disable dropout inside the frozen extractor.
        for layer in self.shared_classifier.modules():
            if isinstance(layer, nn.Dropout):
                layer.p = 0.0

        self.old_head = copy.deepcopy(pretrained_model.classifier[6])

        self.new_head = nn.Sequential(
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_new_classes)
        )

        initialize_linear_layers(self.new_head)

        set_requires_grad(self.features, False)
        set_requires_grad(self.avgpool, False)
        set_requires_grad(self.shared_classifier, False)
        set_requires_grad(self.old_head, False)
        set_requires_grad(self.new_head, True)

    def encode(self, images):
        x = self.features(images)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.shared_classifier(x)
        return x

    def forward(self, images, task):
        x = self.encode(images)

        if task == "imagenet":
            return self.old_head(x)

        if task == "cub":
            return self.new_head(x)

        raise ValueError("task must be either 'imagenet' or 'cub'")


# Step 7 — Evaluation and Supervised Training Helpers

The best checkpoint is selected using the internal CUB validation set.  
Each method still runs for exactly five new-task training epochs.


In [9]:
classification_loss = nn.CrossEntropyLoss()


def trainable_parameters(model):
    return [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]


def current_learning_rate(optimizer):
    return optimizer.param_groups[0]["lr"]


def evaluate_standard_top1(model, dataloader, device, max_batches=None):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(dataloader):
            if max_batches is not None and batch_idx >= max_batches:
                break

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            predictions = logits.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    if total == 0:
        raise RuntimeError("No images were evaluated.")

    return 100.0 * correct / total


def evaluate_task_top1(model, dataloader, device, task, max_batches=None):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for batch_idx, (images, labels) in enumerate(dataloader):
            if max_batches is not None and batch_idx >= max_batches:
                break

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images, task=task)
            predictions = logits.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    if total == 0:
        raise RuntimeError("No images were evaluated.")

    return 100.0 * correct / total


def train_supervised_epoch(model, dataloader, optimizer, device, task="cub"):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        logits = model(images, task=task)
        loss = classification_loss(logits, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def fit_supervised_with_validation(
    model,
    train_loader,
    val_loader,
    device,
    num_epochs,
    learning_rate,
    weight_decay,
    label
):
    optimizer = optim.SGD(
        trainable_parameters(model),
        lr=learning_rate,
        momentum=MOMENTUM,
        weight_decay=weight_decay
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.1,
        patience=SCHEDULER_PATIENCE
    )

    best_val_accuracy = float("-inf")
    best_state = None
    history = []

    for epoch in range(1, num_epochs + 1):
        train_loss, train_accuracy = train_supervised_epoch(
            model=model,
            dataloader=train_loader,
            optimizer=optimizer,
            device=device,
            task="cub"
        )

        val_accuracy = evaluate_task_top1(
            model=model,
            dataloader=val_loader,
            device=device,
            task="cub"
        )

        learning_rate_now = current_learning_rate(optimizer)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "val_accuracy": val_accuracy,
            "learning_rate": learning_rate_now
        })

        print(
            f"{label} | Epoch [{epoch}/{num_epochs}] | "
            f"LR: {learning_rate_now:.6f} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_accuracy:.2f}% | "
            f"Val Acc: {val_accuracy:.2f}%"
        )

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_state = copy.deepcopy(model.state_dict())

        scheduler.step(val_accuracy)

    if best_state is None:
        raise RuntimeError(f"{label}: no checkpoint was saved.")

    model.load_state_dict(best_state)

    return model, pd.DataFrame(history), best_val_accuracy


# Step 8 — LwF Loss and Training Helpers

LwF jointly minimizes:

- CUB classification loss
- ImageNet response-preservation loss

The teacher remains frozen.  
After the one-epoch warm-up, the student shared layers, old head, and new head are jointly optimized.


In [10]:
def distillation_loss(student_old_logits, teacher_old_logits, temperature):
    teacher_probabilities = F.softmax(
        teacher_old_logits / temperature,
        dim=1
    )

    student_log_probabilities = F.log_softmax(
        student_old_logits / temperature,
        dim=1
    )

    return F.kl_div(
        student_log_probabilities,
        teacher_probabilities,
        reduction="batchmean"
    ) * (temperature ** 2)


def train_lwf_epoch(student_model, teacher_model, dataloader, optimizer, device):
    student_model.train()
    teacher_model.eval()

    running_total_loss = 0.0
    running_new_loss = 0.0
    running_old_loss = 0.0

    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.no_grad():
            teacher_old_logits = teacher_model(images)

        student_old_logits, student_new_logits = student_model.forward_both(images)

        loss_new = classification_loss(
            student_new_logits,
            labels
        )

        loss_old = distillation_loss(
            student_old_logits=student_old_logits,
            teacher_old_logits=teacher_old_logits,
            temperature=TEMPERATURE
        )

        total_loss = loss_new + LAMBDA_OLD * loss_old

        total_loss.backward()
        optimizer.step()

        batch_size = images.size(0)

        running_total_loss += total_loss.item() * batch_size
        running_new_loss += loss_new.item() * batch_size
        running_old_loss += loss_old.item() * batch_size

        predictions = student_new_logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return {
        "total_loss": running_total_loss / total,
        "new_loss": running_new_loss / total,
        "old_loss": running_old_loss / total,
        "train_accuracy": 100.0 * correct / total
    }


def fit_lwf_with_validation(
    student_model,
    teacher_model,
    train_loader,
    val_loader,
    device,
    num_epochs,
    learning_rate,
    label
):
    # Jointly optimize shared parameters, old head, and new head.
    set_requires_grad(student_model, True)

    optimizer = optim.SGD(
        trainable_parameters(student_model),
        lr=learning_rate,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.1,
        patience=SCHEDULER_PATIENCE
    )

    best_val_accuracy = float("-inf")
    best_state = None
    history = []

    for epoch in range(1, num_epochs + 1):
        train_metrics = train_lwf_epoch(
            student_model=student_model,
            teacher_model=teacher_model,
            dataloader=train_loader,
            optimizer=optimizer,
            device=device
        )

        val_accuracy = evaluate_task_top1(
            model=student_model,
            dataloader=val_loader,
            device=device,
            task="cub"
        )

        learning_rate_now = current_learning_rate(optimizer)

        history.append({
            "epoch": epoch,
            "total_loss": train_metrics["total_loss"],
            "new_loss": train_metrics["new_loss"],
            "old_loss": train_metrics["old_loss"],
            "train_accuracy": train_metrics["train_accuracy"],
            "val_accuracy": val_accuracy,
            "learning_rate": learning_rate_now
        })

        print(
            f"{label} | Epoch [{epoch}/{num_epochs}] | "
            f"LR: {learning_rate_now:.6f} | "
            f"Total Loss: {train_metrics['total_loss']:.4f} | "
            f"New Loss: {train_metrics['new_loss']:.4f} | "
            f"Old Loss: {train_metrics['old_loss']:.4f} | "
            f"Train Acc: {train_metrics['train_accuracy']:.2f}% | "
            f"Val Acc: {val_accuracy:.2f}%"
        )

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_state = copy.deepcopy(student_model.state_dict())

        scheduler.step(val_accuracy)

    if best_state is None:
        raise RuntimeError(f"{label}: no checkpoint was saved.")

    student_model.load_state_dict(best_state)

    return student_model, pd.DataFrame(history), best_val_accuracy


# Step 9 — Measure the ImageNet Baseline

The pretrained AlexNet model is evaluated once on the first **80 ImageNet validation batches**.


In [11]:
seed_everything(IMAGENET_LOADER_SEED)

baseline_model = create_pretrained_alexnet().to(device)
baseline_model.eval()

baseline_imagenet_accuracy = evaluate_standard_top1(
    model=baseline_model,
    dataloader=imagenet_loader,
    device=device,
    max_batches=IMAGENET_NUM_BATCHES
)

print(
    f"Baseline ImageNet Accuracy on the first "
    f"{IMAGENET_NUM_BATCHES} batches: "
    f"{baseline_imagenet_accuracy:.2f}%"
)

del baseline_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


Baseline ImageNet Accuracy on the first 80 batches: 62.89%


# Step 10 — Run Fine-Tuning, Feature Extraction, and LwF

For the single run with `seed = 42`:

1. Train only the new CUB head for one warm-up epoch.
2. Fine-Tuning starts from the warmed-up state.
3. Feature Extraction is trained separately with frozen ImageNet features.
4. LwF starts from the same warmed-up state used by Fine-Tuning.
5. Each method is trained for five epochs.
6. The best internal-validation checkpoint is evaluated once on:
   - the complete official CUB test set
   - the first 80 ImageNet validation batches


In [12]:
def append_result(
    result_rows,
    seed,
    method,
    old_task_accuracy,
    new_task_accuracy,
    best_val_accuracy
):
    result_rows.append({
        "Seed": seed,
        "Method": method,
        "ImageNet Old Task Accuracy (%)": old_task_accuracy,
        "CUB Official Test Accuracy (%)": new_task_accuracy,
        "Best CUB Validation Accuracy (%)": best_val_accuracy,
        "ImageNet Difference from Baseline (%)": (
            old_task_accuracy - baseline_imagenet_accuracy
        )
    })


all_results = []
all_histories = {}
saved_model_states = {}

for run_seed in RUN_SEEDS:
    print()
    print("=" * 90)
    print(f"STARTING RUN WITH SEED {run_seed}")
    print("=" * 90)

    seed_everything(run_seed)

    pretrained_model = create_pretrained_alexnet().to(device)
    pretrained_model.eval()

    # -------------------------------------------------
    # Stage A: shared one-epoch warm-up
    # -------------------------------------------------

    print()
    print("-" * 90)
    print("STAGE A — Warm-up: train only the new CUB head")
    print("-" * 90)

    warmup_model = DualHeadAlexNet(
        pretrained_model=pretrained_model,
        num_new_classes=NUM_CUB_CLASSES
    ).to(device)

    set_requires_grad(warmup_model, False)
    set_requires_grad(warmup_model.new_head, True)

    warmup_train_loader, warmup_val_loader, _ = make_cub_loaders(
        seed=run_seed
    )

    warmup_model, warmup_history, warmup_best_val = fit_supervised_with_validation(
        model=warmup_model,
        train_loader=warmup_train_loader,
        val_loader=warmup_val_loader,
        device=device,
        num_epochs=WARMUP_EPOCHS,
        learning_rate=HEAD_ONLY_LR,
        weight_decay=WEIGHT_DECAY,
        label=f"Seed {run_seed} | Warm-up"
    )

    warmup_state = copy.deepcopy(warmup_model.state_dict())
    all_histories[(run_seed, "Warm-up")] = warmup_history

    del warmup_train_loader, warmup_val_loader

    # -------------------------------------------------
    # Stage B: Fine-Tuning
    # -------------------------------------------------

    print()
    print("-" * 90)
    print("STAGE B — Fine-Tuning")
    print("-" * 90)

    ft_model = DualHeadAlexNet(
        pretrained_model=pretrained_model,
        num_new_classes=NUM_CUB_CLASSES
    ).to(device)

    ft_model.load_state_dict(warmup_state)

    # Fine-Tuning updates the shared representation and CUB head.
    # The retained ImageNet head remains fixed and is used for evaluation.
    set_requires_grad(ft_model, True)
    set_requires_grad(ft_model.old_head, False)

    ft_train_loader, ft_val_loader, ft_test_loader = make_cub_loaders(
        seed=run_seed + 100
    )

    ft_model, ft_history, ft_best_val = fit_supervised_with_validation(
        model=ft_model,
        train_loader=ft_train_loader,
        val_loader=ft_val_loader,
        device=device,
        num_epochs=FINETUNING_EPOCHS,
        learning_rate=BASE_LR,
        weight_decay=WEIGHT_DECAY,
        label=f"Seed {run_seed} | Fine-Tuning"
    )

    ft_cub_test_accuracy = evaluate_task_top1(
        model=ft_model,
        dataloader=ft_test_loader,
        device=device,
        task="cub"
    )

    ft_imagenet_accuracy = evaluate_task_top1(
        model=ft_model,
        dataloader=imagenet_loader,
        device=device,
        task="imagenet",
        max_batches=IMAGENET_NUM_BATCHES
    )

    append_result(
        result_rows=all_results,
        seed=run_seed,
        method="Fine-Tuning",
        old_task_accuracy=ft_imagenet_accuracy,
        new_task_accuracy=ft_cub_test_accuracy,
        best_val_accuracy=ft_best_val
    )

    all_histories[(run_seed, "Fine-Tuning")] = ft_history

    if SAVE_MODELS:
        saved_model_states[(run_seed, "Fine-Tuning")] = copy.deepcopy(
            ft_model.state_dict()
        )

    print(
        f"Seed {run_seed} | Fine-Tuning | "
        f"ImageNet: {ft_imagenet_accuracy:.2f}% | "
        f"CUB official test: {ft_cub_test_accuracy:.2f}%"
    )

    del ft_model, ft_train_loader, ft_val_loader, ft_test_loader
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # -------------------------------------------------
    # Stage C: Feature Extraction
    # -------------------------------------------------

    print()
    print("-" * 90)
    print("STAGE C — Feature Extraction")
    print("-" * 90)

    seed_everything(run_seed)

    feature_model = FeatureExtractionAlexNet(
        pretrained_model=pretrained_model,
        num_new_classes=NUM_CUB_CLASSES
    ).to(device)

    fe_train_loader, fe_val_loader, fe_test_loader = make_cub_loaders(
        seed=run_seed + 100
    )

    feature_model, fe_history, fe_best_val = fit_supervised_with_validation(
        model=feature_model,
        train_loader=fe_train_loader,
        val_loader=fe_val_loader,
        device=device,
        num_epochs=FEATURE_EXTRACTION_EPOCHS,
        learning_rate=FEATURE_EXTRACTION_LR,
        weight_decay=WEIGHT_DECAY,
        label=f"Seed {run_seed} | Feature Extraction"
    )

    fe_cub_test_accuracy = evaluate_task_top1(
        model=feature_model,
        dataloader=fe_test_loader,
        device=device,
        task="cub"
    )

    # Explicit measurement: no manually inserted baseline value.
    fe_imagenet_accuracy = evaluate_task_top1(
        model=feature_model,
        dataloader=imagenet_loader,
        device=device,
        task="imagenet",
        max_batches=IMAGENET_NUM_BATCHES
    )

    append_result(
        result_rows=all_results,
        seed=run_seed,
        method="Feature Extraction",
        old_task_accuracy=fe_imagenet_accuracy,
        new_task_accuracy=fe_cub_test_accuracy,
        best_val_accuracy=fe_best_val
    )

    all_histories[(run_seed, "Feature Extraction")] = fe_history

    if SAVE_MODELS:
        saved_model_states[(run_seed, "Feature Extraction")] = copy.deepcopy(
            feature_model.state_dict()
        )

    print(
        f"Seed {run_seed} | Feature Extraction | "
        f"ImageNet: {fe_imagenet_accuracy:.2f}% | "
        f"CUB official test: {fe_cub_test_accuracy:.2f}%"
    )

    del feature_model, fe_train_loader, fe_val_loader, fe_test_loader
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # -------------------------------------------------
    # Stage D: Learning without Forgetting
    # -------------------------------------------------

    print()
    print("-" * 90)
    print("STAGE D — Learning without Forgetting")
    print("-" * 90)

    student_model = DualHeadAlexNet(
        pretrained_model=pretrained_model,
        num_new_classes=NUM_CUB_CLASSES
    ).to(device)

    student_model.load_state_dict(warmup_state)

    teacher_model = create_pretrained_alexnet().to(device)
    teacher_model.eval()
    set_requires_grad(teacher_model, False)

    lwf_train_loader, lwf_val_loader, lwf_test_loader = make_cub_loaders(
        seed=run_seed + 100
    )

    student_model, lwf_history, lwf_best_val = fit_lwf_with_validation(
        student_model=student_model,
        teacher_model=teacher_model,
        train_loader=lwf_train_loader,
        val_loader=lwf_val_loader,
        device=device,
        num_epochs=LWF_EPOCHS,
        learning_rate=BASE_LR,
        label=f"Seed {run_seed} | LwF"
    )

    lwf_cub_test_accuracy = evaluate_task_top1(
        model=student_model,
        dataloader=lwf_test_loader,
        device=device,
        task="cub"
    )

    lwf_imagenet_accuracy = evaluate_task_top1(
        model=student_model,
        dataloader=imagenet_loader,
        device=device,
        task="imagenet",
        max_batches=IMAGENET_NUM_BATCHES
    )

    append_result(
        result_rows=all_results,
        seed=run_seed,
        method="LwF",
        old_task_accuracy=lwf_imagenet_accuracy,
        new_task_accuracy=lwf_cub_test_accuracy,
        best_val_accuracy=lwf_best_val
    )

    all_histories[(run_seed, "LwF")] = lwf_history

    if SAVE_MODELS:
        saved_model_states[(run_seed, "LwF")] = copy.deepcopy(
            student_model.state_dict()
        )

    print(
        f"Seed {run_seed} | LwF | "
        f"ImageNet: {lwf_imagenet_accuracy:.2f}% | "
        f"CUB official test: {lwf_cub_test_accuracy:.2f}%"
    )

    del (
        student_model,
        teacher_model,
        pretrained_model,
        warmup_model,
        lwf_train_loader,
        lwf_val_loader,
        lwf_test_loader
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print()
print("All runs completed successfully.")



STARTING RUN WITH SEED 42

------------------------------------------------------------------------------------------
STAGE A — Warm-up: train only the new CUB head
------------------------------------------------------------------------------------------
Seed 42 | Warm-up | Epoch [1/1] | LR: 0.005000 | Train Loss: 5.3850 | Train Acc: 17.56% | Val Acc: 24.75%

------------------------------------------------------------------------------------------
STAGE B — Fine-Tuning
------------------------------------------------------------------------------------------
Seed 42 | Fine-Tuning | Epoch [1/5] | LR: 0.001000 | Train Loss: 2.0972 | Train Acc: 48.29% | Val Acc: 38.88%
Seed 42 | Fine-Tuning | Epoch [2/5] | LR: 0.001000 | Train Loss: 1.4578 | Train Acc: 59.53% | Val Acc: 39.75%
Seed 42 | Fine-Tuning | Epoch [3/5] | LR: 0.001000 | Train Loss: 1.1361 | Train Acc: 67.31% | Val Acc: 43.50%
Seed 42 | Fine-Tuning | Epoch [4/5] | LR: 0.001000 | Train Loss: 0.8839 | Train Acc: 74.09% | Val Acc:

# Step 11 — Results Tables

Because this notebook uses one seed only, the table reports the single completed run.  
The baseline row is included for convenient comparison.


In [13]:
results_df = pd.DataFrame(all_results)

method_order = [
    "Fine-Tuning",
    "Feature Extraction",
    "LwF"
]

results_df["Method"] = pd.Categorical(
    results_df["Method"],
    categories=method_order,
    ordered=True
)

results_df = results_df.sort_values(
    by=["Method", "Seed"]
).reset_index(drop=True)

baseline_row = pd.DataFrame({
    "Seed": ["—"],
    "Method": ["Baseline ImageNet"],
    "ImageNet Old Task Accuracy (%)": [baseline_imagenet_accuracy],
    "CUB Official Test Accuracy (%)": [np.nan],
    "Best CUB Validation Accuracy (%)": [np.nan],
    "ImageNet Difference from Baseline (%)": [0.0]
})

report_table = pd.concat(
    [baseline_row, results_df.astype({"Method": "object"})],
    ignore_index=True
)

print("Final single-run report table")
display(report_table.round(2))


Final single-run report table


,Seed,Method,ImageNet Old Task Accuracy (%),CUB Official Test Accuracy (%),Best CUB Validation Accuracy (%),ImageNet Difference from Baseline (%)
0,—,Baseline ImageNet,62.89,NaN,NaN,0.00
1,42,Fine-Tuning,53.42,45.53,45.00,-9.47
2,42,Feature Extraction,62.89,42.03,42.12,0.00
3,42,LwF,59.49,45.41,45.12,-3.40


# Step 12 — Save Results and Optional Model Weights

The CSV files are useful for the project report and GitHub submission.

Model saving is disabled by default because AlexNet checkpoints are large.  
To save them, set `SAVE_MODELS = True` in Step 0 before running the notebook.


In [14]:
os.makedirs(OUTPUT_ROOT, exist_ok=True)

results_csv_path = os.path.join(
    OUTPUT_ROOT,
    "cub_single_seed_results.csv"
)

report_csv_path = os.path.join(
    OUTPUT_ROOT,
    "cub_report_table.csv"
)

results_df.to_csv(results_csv_path, index=False)
report_table.to_csv(report_csv_path, index=False)

print("Saved:", results_csv_path)
print("Saved:", report_csv_path)

if SAVE_MODELS:
    model_output_root = os.path.join(
        OUTPUT_ROOT,
        "saved_models"
    )

    os.makedirs(model_output_root, exist_ok=True)

    for (seed, method), state_dict in saved_model_states.items():
        safe_method_name = (
            method
            .lower()
            .replace(" ", "_")
            .replace("-", "_")
        )

        save_path = os.path.join(
            model_output_root,
            f"{safe_method_name}_seed_{seed}.pth"
        )

        torch.save(state_dict, save_path)
        print("Saved:", save_path)


Saved: /content/drive/MyDrive/LWF_Project/results_cub_consistent/cub_single_seed_results.csv
Saved: /content/drive/MyDrive/LWF_Project/results_cub_consistent/cub_report_table.csv


# Step 13 — Optional Diagnostic

Feature Extraction freezes the pretrained ImageNet representation and old head.  
Therefore, its ImageNet predictions should agree with the pretrained baseline.

This diagnostic cell is optional. It does not affect the report table.


In [15]:
def compare_old_task_predictions(
    model_a,
    model_b,
    dataloader,
    device,
    max_batches=IMAGENET_NUM_BATCHES
):
    model_a.eval()
    model_b.eval()

    total = 0
    same_predictions = 0

    with torch.no_grad():
        for batch_idx, (images, _) in enumerate(dataloader):
            if max_batches is not None and batch_idx >= max_batches:
                break

            images = images.to(device, non_blocking=True)

            logits_a = model_a(images)
            logits_b = model_b(images, task="imagenet")

            predictions_a = logits_a.argmax(dim=1)
            predictions_b = logits_b.argmax(dim=1)

            same_predictions += (
                predictions_a == predictions_b
            ).sum().item()

            total += images.size(0)

    if total == 0:
        raise RuntimeError("No images were evaluated.")

    print("Compared images:", total)
    print("Identical predictions:", same_predictions)
    print("Agreement:", 100.0 * same_predictions / total, "%")


# Example usage:
#
# diagnostic_baseline = create_pretrained_alexnet().to(device)
# diagnostic_feature_model = FeatureExtractionAlexNet(
#     pretrained_model=diagnostic_baseline,
#     num_new_classes=NUM_CUB_CLASSES
# ).to(device)
#
# compare_old_task_predictions(
#     model_a=diagnostic_baseline,
#     model_b=diagnostic_feature_model,
#     dataloader=imagenet_loader,
#     device=device,
#     max_batches=IMAGENET_NUM_BATCHES
# )
